In [10]:
%pip install pdfplumber
%pip install PyMuPDF
%pip install pyspellchecker

import fitz  # PyMuPDF
import re
import os
from collections import defaultdict
import tkinter as tk
from tkinter import filedialog

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
   ---------------------------------------- 0.0/7.2 MB ? eta -:--:--
   ----------------------- ---------------- 4.2/7.2 MB 35.0 MB/s eta 0:00:01
   ---------------------------------------- 7.2/7.2 MB 26.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


**MULTI_COLUMN + NLP HYPHENATION CLEANSING EXTRACTION**

In [13]:
import fitz  # PyMuPDF
import json
import re
import os
import tkinter as tk
from tkinter import filedialog
from spellchecker import SpellChecker

# Import the multi-column detection utility
from multi_column import column_boxes

# Initialize the English dictionary
spell = SpellChecker()

def fix_hyphenation(match):
    """
    Callback function for RegEx. Checks if the merged word is a valid 
    English word. If yes, it merges them. If no, it separates them with a space.
    """
    word1 = match.group(1)
    word2 = match.group(2)
    combined = (word1 + word2).lower()
    
    if spell.known([combined]):
        return word1 + word2  # e.g., exami + nation -> examination
    else:
        return f"{word1} {word2}" # e.g., age + appropriate -> age appropriate

def select_file():
    root = tk.Tk()
    root.withdraw()
    root.attributes('-topmost', True) 
    
    file_path = filedialog.askopenfilename(
        title="Select the Book (PDF) to Extract",
        filetypes=[("PDF Documents", "*.pdf"), ("All Files", "*.*")]
    )
    
    root.destroy()
    return file_path

def extract_pdf_to_json(pdf_path, output_json_path):
    doc = fitz.open(pdf_path)
    book_data = []

    for page_num in range(len(doc)):
        page = doc.load_page(page_num)
        bboxes = column_boxes(page, footer_margin=50, header_margin=50, no_image_text=True)
        
        page_text = []
        for rect in bboxes:
            raw_text = page.get_text("text", clip=rect, sort=True).strip()
            
            if not raw_text:
                continue
                
            # Replace physical line breaks with spaces. 
            # This turns "exami-\nnation" into "exami- nation" so our smart RegEx can evaluate it.
            raw_text = raw_text.replace('\n', ' ')
            page_text.append(raw_text)
            
        combined_text = "\n\n".join(page_text)
        
        # Apply the smart NLP hyphenation fix across the entire page's text
        combined_text = re.sub(r'(\b[A-Za-z]+)-\s*([A-Za-z]+\b)', fix_hyphenation, combined_text)
        
        # Clean up any residual multi-spaces from the column formatting
        combined_text = re.sub(r' +', ' ', combined_text)

        book_data.append({
            "page_number": page_num + 1,
            "content": combined_text.strip()
        })
        
    with open(output_json_path, 'w', encoding='utf-8') as f:
        json.dump(book_data, f, ensure_ascii=False, indent=4)
        
    print(f"Extraction complete! Saved {len(doc)} pages to: {output_json_path}")

# --- Execution Block ---
selected_pdf_path = select_file()

if selected_pdf_path:
    print(f"Processing: {selected_pdf_path}")
    base_filename = os.path.splitext(os.path.basename(selected_pdf_path))[0]
    output_path = f"{base_filename}_nlp_extracted.json"
    
    extract_pdf_to_json(selected_pdf_path, output_path)
else:
    print("Extraction cancelled. No file was selected.")

Extraction cancelled. No file was selected.


In [ ]:
import re
import os

def extract_chapters_from_txt(txt_file_path, chapter_numbers, output_dir=None):
    """
    Extracts specified chapters from a previously extracted .txt file.
    
    Parameters:
        txt_file_path (str): Path to the full extracted text file.
        chapter_numbers (list): List of chapter numbers to extract (e.g., [1,2,3,10]).
        output_dir (str, optional): Directory to save chapter files. 
                                    If None, saves in the same folder as the input file.
    
    Returns:
        dict: {chapter_number: chapter_text}
    """
    with open(txt_file_path, 'r', encoding='utf-8') as f:
        full_text = f.read()
    
    # Split on chapter headings (case‑insensitive, allowing spaces before/after)
    # Pattern matches: "CHAPTER 1", "CHAPTER 2", etc. at the start of a line.
    # It captures the chapter number.
    parts = re.split(r'\nCHAPTER\s+(\d+)', full_text, flags=re.IGNORECASE)
    
    # The first element is text before the first chapter (ignore it)
    chapters = {}
    for i in range(1, len(parts), 2):
        num = int(parts[i])
        content = parts[i+1] if i+1 < len(parts) else ""
        if num in chapter_numbers:
            chapters[num] = content.strip()
    
    # Optionally save each chapter to a separate .txt file
    if output_dir is None:
        output_dir = os.path.dirname(txt_file_path) or "."
    os.makedirs(output_dir, exist_ok=True)
    
    for num, text in chapters.items():
        chapter_file = os.path.join(output_dir, f"chapter_{num}.txt")
        with open(chapter_file, 'w', encoding='utf-8') as f:
            f.write(text)
        print(f"Saved chapter {num} -> {chapter_file}")
    
    return chapters

# ========== EXAMPLE USAGE ==========
# Run this cell after you have the full extracted .txt file

# Path to the full extracted text file (change to your actual file name)
full_txt_file = "Dentistry for the Child and Adolescent_8th Edition.txt"

# List the chapter numbers you want to extract (e.g., 1,2,3,10)
chapters_to_extract = [1, 2, 3, 10, 13, 18, 30]

# Extract and save them
result = extract_chapters_from_txt(full_txt_file, chapters_to_extract)